# 02 — GOV Forecast Model

**Purpose**: Fit and validate the DASH US Marketplace GOV forecast model.  
**Architecture**: OLS primary, Ridge secondary (robustness).  
**Validation**: Expanding window walk-forward, minimum 8 quarters training.  

See CLAUDE.md §11 for full modeling spec.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
import sys
sys.path.insert(0, '..')

from src.config import MASTER_DF_PATH, OLS_BASE_FEATURES, MODEL_TARGET, CHART_STYLE, COLORS, OUTPUTS_FIGURES
from src.model_gov import load_model_data, compute_baselines, walk_forward_ols, fit_final_ols, forecast_q1_2026

plt.rcParams.update(CHART_STYLE)

## 1. Load Data and Compute Baselines

In [ ]:
df = load_model_data()
df = compute_baselines(df)
print(f"Modeling data: {len(df)} quarters")
print(f"Features available: {[f for f in OLS_BASE_FEATURES if f in df.columns]}")

## 2. Walk-Forward Validation

In [ ]:
pred_df, models = walk_forward_ols(df)
pred_df

In [ ]:
# Exhibit 4: Walk-forward performance
if not pred_df.empty:
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(pred_df['quarter_label'], pred_df['actual_yoy_pct'], 
            color=COLORS['actual'], linewidth=2.5, marker='o', label='Actual GOV YoY %')
    ax.plot(pred_df['quarter_label'], pred_df['predicted_yoy_pct'],
            color=COLORS['dash_primary'], linewidth=2, marker='s', linestyle='--', label='Model predicted')
    
    rmse = np.sqrt((pred_df['abs_error']**2).mean())
    dir_acc = pred_df['direction_correct'].mean()
    ax.set_title(f'Exhibit 4 — Walk-Forward Validation (RMSE={rmse:.1f}pp, Dir. Acc.={dir_acc:.0%})')
    ax.set_ylabel('GOV YoY Growth (%)')
    ax.legend()
    ax.tick_params(axis='x', rotation=45)
    plt.tight_layout()
    fig.savefig(OUTPUTS_FIGURES / 'exhibit4_walkforward.png', dpi=300, bbox_inches='tight')
    plt.show()

## 3. Final OLS Model on All Data

In [ ]:
final_model = fit_final_ols(df)
# Model summary printed by fit_final_ols()

## 4. Q1 2026 Forecast

In [ ]:
forecast = forecast_q1_2026(df, final_model)
forecast

## 5. Ridge Robustness Check (if OLS shows multicollinearity)

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

# Run only if OLS condition number is high (multicollinearity concern)
# VIF check
from statsmodels.stats.outliers_influence import variance_inflation_factor

avail = df.dropna(subset=[MODEL_TARGET] + OLS_BASE_FEATURES)
X = sm.add_constant(avail[[f for f in OLS_BASE_FEATURES if f in avail.columns]])
vif_data = pd.DataFrame({'feature': X.columns,
                          'VIF': [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]})
print("VIF (>10 suggests problematic multicollinearity):")
print(vif_data)